# Kaggriculture submission v10: full-capacity expanded farm

V10 extends v7 by filling all ten daily market hire slots, eliminating the expanded farm's remaining labor bottleneck. The embedded agent is deterministic, stateless, dependency-free, and locally validated against v7 and v6.


In [ ]:
%%writefile main.py


"""Kaggriculture v10: shed-edge livestock routing.

V10 retains v9's feed-safe five-animal expansion and ten-worker crop scheduler,
but moves the added goose and cow from distant NW plots to the unlocked NE shed
edge. The cow can feed, harvest, care, collect fertilizer, and deliver directly
from a shed-access tile; the goose is only one step away. This recovers repeated
travel turns without changing the proven capital, crop, or market policy. The
agent remains deterministic, stateless, and dependency-free at runtime.
"""


PASS = ["PASS"]
MAX_MARKET_ORDERS = 10
FIRST_LAND_COST = 1000
DESIRED_HANDS = 10
FINAL_PLANT_HOUR = 18
SEED_BUY_CUTOFF = 8
LIQUIDATION_DAY = 27
FINAL_CARE_DAY = 28
FEED_TARGET = 21
FEED_REORDER = 10


ANIMALS = {
    "GOOSE": {"cost": 300, "structure": "COOP", "product": "EGG"},
    "COW": {"cost": 400, "structure": "PASTURE", "product": "MILK"},
    "SHEEP": {"cost": 500, "structure": "PASTURE", "product": "WOOL"},
}


# The farmer and first two hands own these roles.  All three structures are at
# most one move from the NW shed-access square (4, 4).
ANIMAL_SLOTS = (
    (4, 4, "GOOSE"),
    (3, 4, "COW"),
    (4, 3, "SHEEP"),
    (5, 3, "GOOSE"),
    (5, 4, "COW"),
)


CROPS = {
    "WHEAT": {
        "seed_cost": 10,
        "base_price": 25,
        "first_yield_day": 2,
        "harvest_day": 4,
        "last_plant_day": 24,
        "ongoing": False,
        "final_age": 4,
    },
    "CARROT": {
        "seed_cost": 20,
        "base_price": 35,
        "first_yield_day": 2,
        "harvest_day": 3,
        "last_plant_day": 25,
        "ongoing": False,
        "final_age": 3,
    },
    "TOMATO": {
        "seed_cost": 50,
        "base_price": 60,
        "first_yield_day": 8,
        "harvest_day": 8,
        "last_plant_day": 18,
        "ongoing": True,
        "final_age": 11,
    },
    "STRAWBERRY": {
        "seed_cost": 100,
        "base_price": 120,
        "first_yield_day": 10,
        "harvest_day": 10,
        "last_plant_day": 13,
        "ongoing": True,
        "final_age": 16,
    },
    "MELON": {
        "seed_cost": 80,
        "base_price": 250,
        "first_yield_day": 10,
        "harvest_day": 10,
        "last_plant_day": 18,
        "ongoing": False,
        "final_age": 10,
    },
}


# Twenty-two crop plots remain.  Five wheat plots are sufficient to replace the
# three units of animal feed consumed each day once their first harvest lands.
CROP_SLOTS = (
    (0, 0, "MELON"), (1, 0, "STRAWBERRY"), (2, 0, "MELON"),
    (3, 0, "WHEAT"), (4, 0, "MELON"),
    (0, 1, "WHEAT"), (1, 1, "MELON"), (2, 1, "STRAWBERRY"),
    (3, 1, "MELON"), (4, 1, "TOMATO"),
    (0, 2, "WHEAT"), (1, 2, "CARROT"), (2, 2, "MELON"),
    (3, 2, "STRAWBERRY"), (4, 2, "MELON"),
    (0, 3, "STRAWBERRY"), (1, 3, "MELON"), (2, 3, "WHEAT"),
    (3, 3, "CARROT"),
    (0, 4, "WHEAT"), (1, 4, "CARROT"), (2, 4, "TOMATO"),
)


# The NE quadrant repeats a market-diversified four-crop pattern.  These crops
# have complementary lead times and price curves, which limits self-inflicted
# gluts while making the 1,000-coin first expansion repay within the season.
EXTRA_PATTERN = ("WHEAT", "TOMATO", "STRAWBERRY", "MELON")
_EXTRA_POINTS = tuple((x, y) for y in range(5) for x in range(5, 10))
CROP_SLOTS = CROP_SLOTS + tuple(
    (x, y, EXTRA_PATTERN[index % len(EXTRA_PATTERN)])
    for index, (x, y) in enumerate(_EXTRA_POINTS)
)
_ANIMAL_POINTS = {(x, y) for x, y, _ in ANIMAL_SLOTS}
CROP_SLOTS = tuple(slot for slot in CROP_SLOTS if slot[:2] not in _ANIMAL_POINTS)


SELL_RULES = {
    # Fertilizer has no town demand, so holding cannot create recovery.
    "FERTILIZER": (12, 1),
    "EGG": (12, 38),
    "MILK": (5, 90),
    "WOOL": (4, 120),
    "MELON": (6, 155),
    "STRAWBERRY": (4, 75),
    "TOMATO": (6, 35),
    "CARROT": (12, 23),
    "WHEAT": (16, 19),
}


SELL_ORDER = (
    "WOOL", "MILK", "MELON", "STRAWBERRY", "FERTILIZER",
    "EGG", "TOMATO", "CARROT", "WHEAT",
)


def _safe_int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def _step_toward(position, target):
    x, y = position
    tx, ty = target
    dx = tx - x
    dy = ty - y
    if abs(dx) >= abs(dy) and dx:
        return ["EAST" if dx > 0 else "WEST"]
    if dy:
        return ["SOUTH" if dy > 0 else "NORTH"]
    return PASS


def _shed_tiles(board_size=10):
    half = board_size // 2
    return (
        (half - 1, half - 1),
        (half, half - 1),
        (half - 1, half),
        (half, half),
    )


def _at_shed(position, board_size=10):
    return tuple(position) in _shed_tiles(board_size)


def _nearest_shed(position, board_size=10):
    x, y = position
    return min(
        _shed_tiles(board_size),
        key=lambda point: (abs(point[0] - x) + abs(point[1] - y), point[1], point[0]),
    )


def _unit_inventory(private, unit_index):
    inventories = private.get("inventories", []) or []
    if 0 <= unit_index < len(inventories) and isinstance(inventories[unit_index], dict):
        return inventories[unit_index]
    return {}


def _tile_at(farm, x, y):
    try:
        return farm["tiles"][y][x]
    except (KeyError, IndexError, TypeError):
        return "LOCKED"


def _animal_counts(farm, private):
    counts = {animal: 0 for animal in ANIMALS}
    for row in (farm.get("tiles", []) or []):
        for tile in row:
            if isinstance(tile, dict) and tile.get("animal") in counts:
                counts[tile["animal"]] += 1
    shed = private.get("shed", {}) or {}
    for animal in counts:
        counts[animal] += max(0, _safe_int(shed.get(animal), 0))
    for inventory in (private.get("inventories", []) or []):
        if not isinstance(inventory, dict):
            continue
        for animal in counts:
            counts[animal] += max(0, _safe_int(inventory.get(animal), 0))
    return counts


def _animal_role_action(farm, private, unit_index, slot, day):
    """Run one persistent animal role, including setup and same-day delivery."""
    x, y, animal = slot
    target = (x, y)
    positions = [tuple(farm.get("farmer", (4, 4)))]
    positions.extend(tuple(pos) for pos in (farm.get("hands", []) or []))
    if unit_index >= len(positions):
        return PASS
    position = positions[unit_index]
    inventory = _unit_inventory(private, unit_index)
    tile = _tile_at(farm, x, y)
    structure = ANIMALS[animal]["structure"]

    live = isinstance(tile, dict) and tile.get("animal") == animal
    if not live:
        carrying_animal = _safe_int(inventory.get(animal), 0) > 0

        # Acquire the animal before leaving the central shed.  Purchases made by
        # the market this turn become available on the following observation.
        if not carrying_animal:
            if _at_shed(position) and _safe_int((private.get("shed", {}) or {}).get(animal), 0) > 0:
                return ["PICKUP", animal, 1]
            return _step_toward(position, _nearest_shed(position))

        # Carry one feed unit during setup so the new animal is fed on placement day.
        if _safe_int(inventory.get("WHEAT"), 0) <= 0 and _at_shed(position):
            if _safe_int((private.get("shed", {}) or {}).get("WHEAT"), 0) > 0:
                return ["PICKUP", "WHEAT", 1]

        if position != target:
            return _step_toward(position, target)
        if tile is None:
            return ["BUILD_COOP" if structure == "COOP" else "BUILD_PASTURE"]
        if isinstance(tile, dict) and tile.get("kind") == structure and "animal" not in tile:
            return ["PLACE", animal]
        if isinstance(tile, dict) and "animal" not in tile:
            return ["DIG"]
        return PASS

    # Specialists only join crop work after finishing their animal route.  If
    # one harvests a crop, send it straight to the shed instead of stranding the
    # load at the animal slot until the automatic end-of-day drop.
    carried_crops = sum(
        max(0, _safe_int(inventory.get(crop), 0)) for crop in CROPS
    )
    if carried_crops > 0 and (day >= 29 or bool(tile.get("fed_today", False))):
        if _at_shed(position):
            return ["DROP"]
        return _step_toward(position, _nearest_shed(position))

    # On the final day, bank every item instead of paying for feed that cannot
    # create another scored production tick.
    carried_output = sum(
        max(0, _safe_int(inventory.get(item), 0)) for item in SELL_RULES
    )
    if day >= 29 and carried_output > 0:
        if _at_shed(position):
            return ["DROP"]
        return _step_toward(position, _nearest_shed(position))

    if position != target:
        # Fetch feed first when needed; otherwise return to the assigned animal.
        if day <= FINAL_CARE_DAY and not bool(tile.get("fed_today", False)):
            if _safe_int(inventory.get("WHEAT"), 0) <= 0:
                if _at_shed(position):
                    if _safe_int((private.get("shed", {}) or {}).get("WHEAT"), 0) > 0:
                        return ["PICKUP", "WHEAT", 1]
                else:
                    return _step_toward(position, _nearest_shed(position))
        return _step_toward(position, target)

    if day <= FINAL_CARE_DAY and not bool(tile.get("fed_today", False)):
        if _safe_int(inventory.get("WHEAT"), 0) > 0:
            return ["FEED"]
        if _at_shed(position) and _safe_int((private.get("shed", {}) or {}).get("WHEAT"), 0) > 0:
            return ["PICKUP", "WHEAT", 1]
        return _step_toward(position, _nearest_shed(position))

    # Harvest before care/collection so capped animal output cannot block the
    # following night's production.
    if _safe_int(tile.get("yield_units"), 0) > 0:
        return ["HARVEST"]
    if day <= FINAL_CARE_DAY and not bool(tile.get("cared_today", False)):
        return ["CARE"]
    if bool(tile.get("fertilizer_available", False)):
        return ["COLLECT_FERTILIZER"]

    carried_output = sum(
        max(0, _safe_int(inventory.get(item), 0))
        for item in ("EGG", "MILK", "WOOL", "FERTILIZER")
    )
    if carried_output > 0:
        if _at_shed(position):
            return ["DROP"]
        return _step_toward(position, _nearest_shed(position))
    return PASS


def _animal_is_done(farm, private, unit_index, slot, day):
    """Return True once this specialist can safely help with crops today."""
    x, y, animal = slot
    tile = _tile_at(farm, x, y)
    if not (isinstance(tile, dict) and tile.get("animal") == animal):
        return False
    inventory = _unit_inventory(private, unit_index)
    if any(_safe_int(inventory.get(item), 0) > 0 for item in SELL_RULES):
        return False
    if day <= FINAL_CARE_DAY and not bool(tile.get("fed_today", False)):
        return False
    if _safe_int(tile.get("yield_units"), 0) > 0:
        return False
    if day <= FINAL_CARE_DAY and not bool(tile.get("cared_today", False)):
        return False
    if bool(tile.get("fertilizer_available", False)):
        return False
    return True


def _crop_task(tile, crop, day, hour):
    data = CROPS[crop]
    if tile is None:
        if day <= data["last_plant_day"] and hour <= FINAL_PLANT_HOUR:
            return 4, ["PLANT", crop]
        return None
    if tile == "LOCKED" or not isinstance(tile, dict):
        return None
    if tile.get("kind") == "WEED":
        return 3, ["DIG"]
    if tile.get("kind") != "PLANT":
        return None

    actual = tile.get("crop")
    actual_data = CROPS.get(actual)
    if actual_data is None:
        return 3, ["DIG"]
    age = day - _safe_int(tile.get("planted_day"), day)
    held = _safe_int(tile.get("yield_units"), 0)
    watered = bool(tile.get("watered_today", False))

    if day >= 28 and held > 0 and age >= actual_data["first_yield_day"]:
        return 0, ["HARVEST"]
    if actual_data["ongoing"]:
        if held >= 4 or (age >= actual_data["final_age"] and held > 0):
            return 0, ["HARVEST"]
        if age > actual_data["final_age"] and held <= 0:
            return 3, ["DIG"]
        if not watered:
            return 1, ["WATER"]
        return None

    if held > 0 and age >= actual_data["harvest_day"]:
        if age == actual_data["harvest_day"] and not watered:
            return 0, ["WATER"]
        return 0, ["HARVEST"]
    if not watered:
        return 1, ["WATER"]
    return None


def _crop_tasks(farm, private, day, hour):
    seeds = {
        crop: _safe_int((private.get("seeds", {}) or {}).get(crop), 0)
        for crop in CROPS
    }
    tasks = []
    for x, y, crop in CROP_SLOTS:
        task = _crop_task(_tile_at(farm, x, y), crop, day, hour)
        if task is None:
            continue
        priority, action = task
        if action[0] == "PLANT":
            if seeds[crop] <= 0:
                continue
            seeds[crop] -= 1
        tasks.append({"priority": priority, "target": (x, y), "action": action})
    return tasks


def _assign_crop_actions(farm, private, day, hour, positions, unit_indices):
    actions = {}
    tasks = _crop_tasks(farm, private, day, hour)
    remaining = set(unit_indices)
    while tasks and remaining:
        choices = []
        for unit_index in remaining:
            ux, uy = positions[unit_index]
            for task_index, task in enumerate(tasks):
                tx, ty = task["target"]
                distance = abs(tx - ux) + abs(ty - uy)
                choices.append(
                    (
                        task["priority"], distance, ty, tx, unit_index, task_index,
                    )
                )
        _, _, _, _, unit_index, task_index = min(choices)
        task = tasks.pop(task_index)
        remaining.remove(unit_index)
        if positions[unit_index] == task["target"]:
            actions[unit_index] = task["action"]
        else:
            actions[unit_index] = _step_toward(positions[unit_index], task["target"])
    return actions


def _assign_unit_actions(farm, private, day, hour):
    positions = [tuple(farm.get("farmer", (4, 4)))]
    positions.extend(tuple(pos) for pos in (farm.get("hands", []) or []))
    actions = [PASS for _ in positions]

    animal_role_count = min(len(ANIMAL_SLOTS), len(positions))
    crop_indices = []
    for unit_index in range(animal_role_count):
        slot = ANIMAL_SLOTS[unit_index]
        if _animal_is_done(farm, private, unit_index, slot, day):
            crop_indices.append(unit_index)
        else:
            actions[unit_index] = _animal_role_action(
                farm, private, unit_index, slot, day
            )

    for unit_index in range(animal_role_count, len(positions)):
        inventory = _unit_inventory(private, unit_index)
        carried = sum(
            max(0, _safe_int(inventory.get(item), 0))
            for item in SELL_RULES
        )
        if day >= 29 and carried > 0:
            if _at_shed(positions[unit_index]):
                actions[unit_index] = ["DROP"]
            else:
                actions[unit_index] = _step_toward(
                    positions[unit_index], _nearest_shed(positions[unit_index])
                )
        else:
            crop_indices.append(unit_index)

    for unit_index, action in _assign_crop_actions(
        farm, private, day, hour, positions, crop_indices
    ).items():
        actions[unit_index] = action
    return actions


def _carried_totals(private):
    totals = {}
    for inventory in (private.get("inventories", []) or []):
        if not isinstance(inventory, dict):
            continue
        for item, quantity in inventory.items():
            totals[item] = totals.get(item, 0) + max(0, _safe_int(quantity))
    return totals


def _predicted_drop(private, unit_actions):
    totals = {}
    inventories = private.get("inventories", []) or []
    for index, action in enumerate(unit_actions):
        if not (isinstance(action, list) and action and action[0] == "DROP"):
            continue
        if index >= len(inventories) or not isinstance(inventories[index], dict):
            continue
        for item, quantity in inventories[index].items():
            totals[item] = totals.get(item, 0) + max(0, _safe_int(quantity))
    return totals


def _sell_orders(private, market, day, predicted_drop, limit):
    if limit <= 0:
        return []
    shed = private.get("shed", {}) or {}
    prices = (market or {}).get("prices", {}) or {}
    carried = _carried_totals(private)
    exposure = sum(max(0, _safe_int(v)) for v in shed.values()) + sum(carried.values())
    terminal = day >= LIQUIDATION_DAY
    forced = exposure >= 78
    orders = []

    for item in SELL_ORDER:
        dropping = _safe_int(predicted_drop.get(item), 0)
        held = _safe_int(shed.get(item), 0) + dropping
        if item == "WHEAT":
            carried_wheat = _safe_int(carried.get("WHEAT"), 0)
            non_dropping = max(0, carried_wheat - dropping)
            total_wheat = held + non_dropping
            reserve = FEED_TARGET if day < 29 else 0
            held = min(held, max(0, total_wheat - reserve))
        if held <= 0:
            continue
        batch, floor = SELL_RULES[item]
        price = _safe_int(prices.get(item), 0)
        if terminal or forced or price >= floor:
            quantity = held if terminal else min(held, batch)
            orders.append(["SELL", item, quantity])
            if len(orders) >= limit:
                break
    return orders


def _planned_seed_needs(farm, private, day, hour):
    wanted = {crop: 0 for crop in CROPS}
    for x, y, crop in CROP_SLOTS:
        tile = _tile_at(farm, x, y)
        last_day = CROPS[crop]["last_plant_day"]
        before_cutoff = day < last_day or (
            day == last_day and hour < SEED_BUY_CUTOFF
        )
        if before_cutoff and (
            tile is None or (isinstance(tile, dict) and tile.get("kind") == "WEED")
        ):
            wanted[crop] += 1
    seeds = private.get("seeds", {}) or {}
    return {
        crop: max(0, wanted[crop] - _safe_int(seeds.get(crop), 0))
        for crop in CROPS
    }


def _procurement_orders(farm, private, day, hour, slots):
    if slots <= 0:
        return []
    orders = []
    cash = float(farm.get("money", 0))
    carried = _carried_totals(private)
    shed = private.get("shed", {}) or {}

    total_wheat = _safe_int(shed.get("WHEAT"), 0) + _safe_int(carried.get("WHEAT"), 0)
    wheat_price = max(
        1,
        _safe_int(
            ((farm.get("_market", {}) or {}).get("prices", {}) or {}).get("WHEAT"),
            30,
        ),
    )
    # Stage herd growth one animal per species per turn and preserve enough cash
    # to establish the original trio's feed buffer. This prevents expansion
    # capital from starving newly placed livestock before the first crop sale.
    feed_cash_floor = 80 + max(0, FEED_TARGET - total_wheat) * max(30, wheat_price + 4)
    if day <= 18:
        counts = _animal_counts(farm, private)
        desired = {
            animal: sum(1 for _, _, kind in ANIMAL_SLOTS if kind == animal)
            for animal in ANIMALS
        }
        for animal in ("GOOSE", "COW", "SHEEP"):
            missing = desired[animal] - counts[animal]
            cost = ANIMALS[animal]["cost"]
            if (
                missing > 0
                and len(orders) < slots
                and cash >= cost + feed_cash_floor
            ):
                orders.append(["BUY_ANIMAL", animal, 1])
                cash -= cost


    if day <= FINAL_CARE_DAY and total_wheat <= FEED_REORDER and len(orders) < slots:
        price = max(1, _safe_int(((farm.get("_market", {}) or {}).get("prices", {}) or {}).get("WHEAT"), 30))
        quantity = FEED_TARGET - total_wheat
        affordable = max(0, int((cash - 100) // max(30, price + 4)))
        quantity = min(quantity, affordable)
        if quantity > 0:
            orders.append(["BUY_PRODUCT", "WHEAT", quantity])
            cash -= quantity * max(30, price + 4)

    if day >= 26 or len(orders) >= slots:
        return orders[:slots]
    needs = _planned_seed_needs(farm, private, day, hour)
    for crop in ("WHEAT", "CARROT", "MELON", "TOMATO", "STRAWBERRY"):
        if len(orders) >= slots:
            break
        quantity = needs[crop]
        if quantity <= 0:
            continue
        cost = CROPS[crop]["seed_cost"]
        affordable = max(0, int((cash - 80) // cost))
        quantity = min(quantity, affordable)
        if quantity <= 0:
            continue
        orders.append(["BUY_SEED", crop, quantity])
        cash -= quantity * cost
    return orders[:slots]


def _market_orders(farm, private, market, day, hour, predicted_drop):
    buy_land = (
        hour == 1
        and len(farm.get("unlocked_quadrants", []) or []) == 1
        and float(farm.get("money", 0)) >= FIRST_LAND_COST + 80
    )

    # Reserve expansion cash before sizing purchases instead of depending on
    # partial fills after BUY_LAND consumes the first 1,000 coins.
    farm_for_buying = dict(farm)
    farm_for_buying["_market"] = market or {}
    if buy_land:
        farm_for_buying["money"] = (
            float(farm.get("money", 0)) - FIRST_LAND_COST
        )

    current_hands = len(farm.get("hands", []) or [])
    needed_hires = (
        max(0, DESIRED_HANDS - current_hands)
        if hour == 0 and day <= 29 else 0
    )
    reserved_slots = needed_hires + int(buy_land)
    sale_limit = max(0, MAX_MARKET_ORDERS - reserved_slots)
    orders = _sell_orders(private, market, day, predicted_drop, sale_limit)

    for _ in range(needed_hires):
        orders.append(["HIRE"])

    free = MAX_MARKET_ORDERS - len(orders) - int(buy_land)
    if free > 0:
        orders.extend(
            _procurement_orders(farm_for_buying, private, day, hour, free)
        )

    if buy_land:
        orders.insert(0, ["BUY_LAND"])
    return orders[:MAX_MARKET_ORDERS]


def agent(obs):
    """Required Kaggle entrypoint."""
    try:
        farms = obs.get("farms", []) or []
        player = _safe_int(obs.get("player"), 0)
        private = obs.get("private", {}) or {}
        if player < 0 or player >= len(farms):
            return {"farmer": PASS, "hands": [], "market": []}
        farm = farms[player]
        day = _safe_int(obs.get("day"), 0)
        hour = _safe_int(obs.get("hour"), 0)
        market = obs.get("market", {}) or {}

        unit_actions = _assign_unit_actions(farm, private, day, hour)
        predicted = _predicted_drop(private, unit_actions)
        market_orders = _market_orders(farm, private, market, day, hour, predicted)
        return {
            "farmer": unit_actions[0] if unit_actions else PASS,
            "hands": unit_actions[1:],
            "market": market_orders,
        }
    except Exception:
        hands = []
        try:
            farms = obs.get("farms", []) or []
            player = _safe_int(obs.get("player"), 0)
            if 0 <= player < len(farms):
                hands = [PASS for _ in (farms[player].get("hands", []) or [])]
        except Exception:
            hands = []
        return {"farmer": PASS, "hands": hands, "market": []}


